### Numpy array vs Python lists

# 🔬 Advanced NumPy — Speed, Indexing, Broadcast

**Topic:** 06.02 · **Level:** 🟠 Intermediate · **Type:** THEORY + CODE

## 📖 Numpy vs Python lists — speed

```python
import numpy as np
import time

# Python list addition (loop)
c = [x + y for x, y in zip(a, b)]

# Numpy vectorized
a = np.arange(10000000)
c = a + b
```

- Numpy **C/CPU optimized** — pure loop se ~50-100x fast
- Ye code me `time.time()` se measure hota hai
- Vectorized operation ek hi pass me, loop nie karta
- Data science me bade arrays pe yehi farak padta hai

## 🔬 Deep Dive : Speed — vectorization wins

```python
# list approach
a = [i for i in range(10000000)]
b = [i for i in range(10000000)]
# manual loop... slow

# numpy
import numpy as np
a = np.arange(10000000)
b = np.arange(10000000)
# a + b one call — C speed
```

- Vectorized ops run at C speed (SIMD + no Python loop)
- Same operation: numpy ~50-100x faster than Python list loop
- Reason: tight C loop + contiguous memory
- Python per-element creates interpreter overhead
- Approx 3.26s vs 0.06s (~54x)
- CPU: numpy uses optimized BLAS/LAPACK in many ops
- Rule: never element-loop when vector op exists
- This scale: 10M element addition instant
- ML data millions row — numpy is the backbone

In [60]:
# speed
# list
a = [i for i in range(10000000)]
b = [i for i in range(10000000,20000000)]

c = []
import time 

start = time.time()
for i in range(len(a)):
  c.append(a[i] + b[i])
print(time.time()-start)

3.2699835300445557


In [61]:
# numpy
import numpy as np
a = np.arange(10000000)
b = np.arange(10000000,20000000)

start = time.time()
c = a + b
print(time.time()-start)

0.06481003761291504


In [62]:
3.26/0.06

54.33333333333333

## 📖 Memory usage

```python
import sys
sys.getsizeof([...])          # list: huge
sys.getsizeof(np.arange(..., dtype=np.int8))  # numpy: compact
```

- List — har element khud object (overhead bhot)
- Numpy array — contiguous memory, ek type
- `dtype=np.int8` → 1 byte per element vs list me 28+ bytes
- `sys.getsizeof()` compare karo — numpy farak aata hai

## 🔬 Deep Dive : Memory efficiency

```python
# list: each element a Python int object + pointer (~28 bytes)
import sys
sys.getsizeof([i for i in range(10000000)])   # ~80MB+

# numpy int8: 1 byte per element, contiguous
a = np.arange(10000000, dtype=np.int8)
sys.getsizeof(a)     # ~10MB (data only)
```

- List: pointer (8) + int object (~28) + list overhead — huge
- Numpy: raw C array tight packed — minimum
- dtype choice = memory control (int8 min, float64 8 bytes)
- int8 → 1 byte; float32 4; float64 8
- Contiguous = cache-friendly fast
- numpy memory ~= size * itemsize only
- Efficiency: store 10M ints → MB not GB
- Convenience: homogeneous type assumption
- ML models train on (N, features) numpy — compact

In [63]:
# memory
a = [i for i in range(10000000)]
import sys

sys.getsizeof(a)

81528048

In [67]:
a = np.arange(10000000,dtype=np.int8)
sys.getsizeof(a)

10000104

In [ ]:
# convenience

### Advanced Indexing

In [77]:
# Normal Indexing and slicing

a = np.arange(24).reshape(6,4)
a

array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11],
       [12, 13, 14, 15],
       [16, 17, 18, 19],
       [20, 21, 22, 23]])

In [70]:
a[1,2]

5

In [71]:
a[1:3,1:3]

array([[4, 5],
       [7, 8]])

## 📖 Fancy indexing

```python
a = np.arange(24).reshape(6, 4)
a[:, [0, 2, 3]]    # specific columns select (list se)
```

- Normal slice — range se (`1:3`)
- **Fancy** — list/array of indices jaise `[0, 2, 3]`
- Kisi bhi order me indices dene se sorted, repeated allowed
- Result naye array me copy hota hai (view nahi)

## 🔬 Deep Dive : Fancy indexing — index arrays

```python
a = np.arange(24).reshape(6,4)
a[:,[0,2,3]]      # select specific COLUMNS (0,2,3) — copy
a[[1,2,0],:]      # specific rows
```

- Fancy indexing: index/pick arbitrary positions with INDEX ARRAYS
- Result is a **COPY** (unlike slicing views)
- List/array of indices → gather elements
- Row/col: `a[rows, cols]` tag cross
- Boolean masks alongside (next)
- Use: reorder rows, subsample, feature select
- `a[[0,1],[1,2]]` → (a[0,1], a[1,2]) pair-wise
- Different from slices — fancy = copy semantics
- DataFrame `.iloc` same concept later
- Powerful data extraction tool

In [81]:
# Fancy Indexing

a[:,[0,2,3]]

array([[ 0,  2,  3],
       [ 4,  6,  7],
       [ 8, 10, 11],
       [12, 14, 15],
       [16, 18, 19],
       [20, 22, 23]])

## 📖 Boolean / mask indexing

```python
a = np.random.randint(1, 100, 24).reshape(6, 4)

a[a > 50]                    # saare > 50
a[a % 2 == 0]                # even numbers
a[(a > 50) & (a % 2 == 0)]   # dono conditions
a[~(a % 7 == 0)]             # not divisible by 7
```

- Condition array of True/False — **mask**
- `a[mask]` — sirf wahi elements jaha True
- Combine: `&` and, `|` or, `~` not (python and/or nahi!)
- Data filtering ka sabse shur tarika — SQL WHERE jaisa

## 🔬 Deep Dive : Boolean / mask indexing

```python
a = np.random.randint(1,100,24).reshape(6,4)
a > 50                          # bool mask same shape
a[a > 50]                       # values where True (1D result)
a[a % 2 == 0]                   # evens
a[(a > 50) & (a % 2 == 0)]      # combine conditions with &
a[~(a % 7 == 0)]                # not divisible by 7
```

- Boolean mask = same-shape bool array
- `a[mask]` → elements where mask True (1D flat result)
- Conditions produce mask: `a>50`, `a%2==0`
- Combine: `&` (and), `|` (or), `~` (not) — NOT python and/or
- Parentheses required around conditions
- Masking = most common filtering tool
- Write: `a[(a>50) & (a<90)]` ranged
- Sets membership by mask (np.isin aage)
- Result always 1D flattened selection
- Critical: & not and — numpy error otherwise

In [88]:
# Boolean Indexing
a = np.random.randint(1,100,24).reshape(6,4)
a

array([[76, 98, 99, 39],
       [91, 46, 88, 23],
       [45,  6, 83,  1],
       [37, 43, 78, 85],
       [54, 73, 61, 53],
       [40, 93, 85, 77]])

In [90]:
# find all numbers greater than 50
a[a > 50]

array([76, 98, 99, 91, 88, 83, 78, 85, 54, 73, 61, 53, 93, 85, 77])

In [92]:
# find out even numbers
a[a % 2 == 0]

array([76, 98, 46, 88,  6, 78, 54, 40])

In [97]:
# find all numbers greater than 50 and are even

a[(a > 50) & (a % 2 == 0)]

ValueError: ignored

In [96]:
# find all numbers not divisible by 7
a[~(a % 7 == 0)]

array([76, 99, 39, 46, 88, 23, 45,  6, 83,  1, 37, 43, 78, 85, 54, 73, 61,
       53, 40, 93, 85])

### Broadcasting

The term broadcasting describes how NumPy treats arrays with different shapes during arithmetic operations.

The smaller array is “broadcast” across the larger array so that they have compatible shapes.

## 📖 Broadcasting — same shape

```python
a = np.arange(6).reshape(2, 3)     # (2,3)
b = np.arange(6, 12).reshape(2, 3) # (2,3)
a + b   # elementwise direct
```

- Same shape → direct elementwise, koi magic nahi
- Broadcast ka khel tab hota hai jab shapes DIFFER

## 🔬 Deep Dive : Broadcasting — same shape

```python
a = np.arange(6).reshape(2,3)
b = np.arange(6,12).reshape(2,3)
a + b        # elementwise — shapes match directly
```

- Broadcasting = elementwise ops between compatible shapes
- Same shape → direct elementwise, no copy
- Compares shape dim-by-dim
- Rule 1: same dims → fine
- a + b → c[i,j] = a[i,j]+b[i,j]
- Vectorized — fast
- Diff shape needs broadcast rule (aage)
- Matrices: shapes equal or 1 / missing
- Alignment: trailing dims first
- Works for + - * / ** etc.
- numba-vectorized intuitive

In [99]:
# same shape
a = np.arange(6).reshape(2,3)
b = np.arange(6,12).reshape(2,3)

print(a)
print(b)

print(a+b)

[[0 1 2]
 [3 4 5]]
[[ 6  7  8]
 [ 9 10 11]]
[[ 6  8 10]
 [12 14 16]]


## 📖 Broadcasting — diff shape

```python
a = np.arange(6).reshape(2, 3)     # (2,3)
b = np.arange(3).reshape(1, 3)     # (1,3)
a + b   # b ko a jitna stretch karke add!
```

- Numpy smaller array ko **stretch** karke bada banata hai
- `(1,3)` ka `b` → rows replicate hoke `(2,3)` ban jata hai
- Isse loop likhne ki zaroorat nahi — operation vectorized
- Broadcasting tabhi hoga jab dims compatible hon

## 🔬 Deep Dive : Broadcasting — different shapes

```python
a = np.arange(6).reshape(2,3)   # (2,3)
b = np.arange(3).reshape(3)     # (3,)
a + b        # b stretches along rows → each row + b
```

- a (2,3) + b (3,) → b broadcast to (2,3)
- Trailing dimensions align; dims of size 1 stretch
- Rule 2: dims either equal or one of them is 1
- Rules summary:
  1. align trailing dims
  2. minus-minus: equal | 1 stretches | missing treated 1
  3. result shape = max dims
- Examples:
  - (3,4) + (4,) ✓
  - (3,4) + (1,4) ✓
  - (3,1)+(1,4) → (3,4) ✓
  - (3,4)+(4,3) ✗ mismatch
  - (4,4)+(2,2) ✗
- Scalar broadcast: (2,3)+5 → all
- Concept: no explicit loops — implicit replication
- Widely used w/ mean subtract, scaling, neural nets bias

In [101]:
# diff shape
a = np.arange(6).reshape(2,3)
b = np.arange(3).reshape(1,3)

print(a)
print(b)

print(a+b)

[[0 1 2]
 [3 4 5]]
[[0 1 2]]
[[0 2 4]
 [3 5 7]]


#### Broadcasting Rules

**1. Make the two arrays have the same number of dimensions.**<br>
- If the numbers of dimensions of the two arrays are different, add new dimensions with size 1 to the head of the array with the smaller dimension.<br>

**2. Make each dimension of the two arrays the same size.**<br>
- If the sizes of each dimension of the two arrays do not match, dimensions with size 1 are stretched to the size of the other array.
- If there is a dimension whose size is not 1 in either of the two arrays, it cannot be broadcasted, and an error is raised.

<img src = "https://jakevdp.github.io/PythonDataScienceHandbook/figures/02.05-broadcasting.png">

## 📖 Broadcasting rules quick reference

```python
a = np.arange(12).reshape(4, 3)   # (4,3)
b = np.arange(3)                  # (3,) → (1,3) → (4,3)
a + b

a = np.arange(12).reshape(3, 4)   # (3,4)
b = np.arange(3)                  # (3,) → can't (4 vs 3) → ERROR
```

2 rules:
1. Right-align shapes; har dim me `1` ya equals hona chahiye
2. Dims `1` wale stretch hote hain

| Shape | Compatible? |
|---|---|
| (4,3) + (3,) | ✅ |
| (3,4) + (3,) | ❌ |
| (1,3) + (3,1) | ✅ (→ (3,3)) |
| (4,4) + (2,2) | ❌ |

In [103]:
# More examples

a = np.arange(12).reshape(4,3)
b = np.arange(3)

print(a)
print(b)

print(a+b)

[[ 0  1  2]
 [ 3  4  5]
 [ 6  7  8]
 [ 9 10 11]]
[0 1 2]
[[ 0  2  4]
 [ 3  5  7]
 [ 6  8 10]
 [ 9 11 13]]


In [104]:
a = np.arange(12).reshape(3,4)
b = np.arange(3)

print(a)
print(b)

print(a+b)

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
[0 1 2]


ValueError: ignored

In [105]:
a = np.arange(3).reshape(1,3)
b = np.arange(3).reshape(3,1)

print(a)
print(b)

print(a+b)

[[0 1 2]]
[[0]
 [1]
 [2]]
[[0 1 2]
 [1 2 3]
 [2 3 4]]


In [107]:
a = np.arange(3).reshape(1,3)
b = np.arange(4).reshape(4,1)

print(a)
print(b)

print(a + b)

[[0 1 2]]
[[0]
 [1]
 [2]
 [3]]
[[0 1 2]
 [1 2 3]
 [2 3 4]
 [3 4 5]]


In [108]:
a = np.array([1])
# shape -> (1,1)
b = np.arange(4).reshape(2,2)
# shape -> (2,2)

print(a)
print(b)

print(a+b)

[1]
[[0 1]
 [2 3]]
[[1 2]
 [3 4]]


In [109]:
a = np.arange(12).reshape(3,4)
b = np.arange(12).reshape(4,3)

print(a)
print(b)

print(a+b)

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
[[ 0  1  2]
 [ 3  4  5]
 [ 6  7  8]
 [ 9 10 11]]


ValueError: ignored

In [110]:
a = np.arange(16).reshape(4,4)
b = np.arange(4).reshape(2,2)

print(a)
print(b)

print(a+b)

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]
 [12 13 14 15]]
[[0 1]
 [2 3]]


ValueError: ignored

### Working with mathematical formulas

In [114]:
a = np.arange(10)
np.sin(a)

array([ 0.        ,  0.84147098,  0.90929743,  0.14112001, -0.7568025 ,
       -0.95892427, -0.2794155 ,  0.6569866 ,  0.98935825,  0.41211849])

## 📖 Math formulas — vectorized

```python
def sigmoid(array):
    return 1 / (1 + np.exp(-(array)))

a = np.arange(100)
sigmoid(a)

def mse(actual, predicted):
    return np.mean((actual - predicted) ** 2)
```

- ML me har formula array pe vectorized likha jata hai
- sigmoid, MSE, BCE — `np.exp`, `np.mean`, elementwise ka combo
- Loop ki jagah direct array math — 1000x clean + fast
- Binary cross-entropy bhi `np.mean((a - p)**2)` se similar

## 🔬 Deep Dive : Math formulas — vectorized

```python
def sigmoid(array):
    return 1 / (1 + np.exp(-array))     # vectorized! elementwise

def mse(actual, predicted):
    return np.mean((actual - predicted) ** 2)   # whole arrays
```

- ML formulas = numpy one-liners (no loops)
- sigmoid: e^-x vectorized over array
- MSE: diff → square → mean — all array ops
- Binary cross entropy similar
- np.array ops handle any shape
- Implicit broadcasting int enrichment
- These ARE model math functions — real usage
- Vectorization = numpy codebase style
- Larger scale: same functions on matrices
- Prefer np ops over manual loops in ML code

In [117]:
# sigmoid
def sigmoid(array):
  return 1/(1 + np.exp(-(array)))


a = np.arange(100)

sigmoid(a)

array([0.5       , 0.73105858, 0.88079708, 0.95257413, 0.98201379,
       0.99330715, 0.99752738, 0.99908895, 0.99966465, 0.99987661,
       0.9999546 , 0.9999833 , 0.99999386, 0.99999774, 0.99999917,
       0.99999969, 0.99999989, 0.99999996, 0.99999998, 0.99999999,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.     

In [118]:
# mean squared error

actual = np.random.randint(1,50,25)
predicted = np.random.randint(1,50,25)

In [122]:
def mse(actual,predicted):
  return np.mean((actual - predicted)**2)

mse(actual,predicted)

500.12

In [125]:
# binary cross entropy
np.mean((actual - predicted)**2)

500.12

In [119]:
actual

array([ 5,  3,  9,  7,  3, 36, 49, 28, 20, 40,  2, 23, 29, 18, 30, 23,  7,
       40, 15, 11, 27, 44, 32, 28, 10])

### Working with missing values

## 📖 Missing values — np.nan

```python
a = np.array([1, 2, 3, 4, np.nan, 6])
a[~np.isnan(a)]     # nan hatao
```

- `np.nan` = Not-a-Number — missing data ka marker
- `np.isnan(a)` — mask: jaha NaN True
- `~np.isnan(a)` — flip: jaha NOT NaN
- `a[mask]` — sirf valid values (nan-free clean array)
- Typical data-cleaning step in ML pipelines

## 🔬 Deep Dive : Missing values — np.nan

```python
a = np.array([1,2,3,4,np.nan,5,6])
a[~np.isnan(a)]      # drop nan → array([1,2,3,4,5,6])
```

- **np.nan** — Not a Number (missing marker)
- nan propagates: any op with nan → nan (careful)
- Detect: `np.isnan(a)` → bool mask
- Remove/filter: `a[~np.isnan(a)]`
- Functions with nan: np.nanmean, np.nansum, etc.
- nan replaces None in numeric arrays (float only)
- int arrays can't hold nan (dtype object needed)
- Fill: np.nan_to_num(x, nan=0)
- DataFrame uses NaN too (pandas) — cornerstone
- Check any: `np.isnan(a).any()`

In [126]:
# Working with missing values -> np.nan
a = np.array([1,2,3,4,np.nan,6])
a

array([ 1.,  2.,  3.,  4., nan,  6.])

In [130]:
a[~np.isnan(a)]

array([1., 2., 3., 4., 6.])

### Plotting Graphs

## 📖 Plotting with numpy + matplotlib

```python
import matplotlib.pyplot as plt
import numpy as np

x = np.linspace(-10, 10, 100)   # 100 points
y = x ** 2                        # vectorized formula
plt.plot(x, y)
```

- `np.linspace` — smooth x points
- Any formula → vectorized → `plt.plot(x, y)`
- `y = x`, `y = x**2`, `y = np.sin(x)`, `y = x*np.log(x)`, sigmoid
- Numpy + matplotlib = visual math

## 🔬 Deep Dive : Plotting with numpy + matplotlib

```python
import matplotlib.pyplot as plt
x = np.linspace(-10,10,100)      # 100 points grid
y = x**2
plt.plot(x, y)
plt.show()

y = np.sin(x)                     # wave
y = x*np.log(x)
y = 1/(1+np.exp(-x))              # sigmoid curve
```

- matplotlib consumes numpy arrays directly
- linspace — smooth curves (100+ points)
- Visualize function shapes: parabola, sine, logistic
- plt.plot(x,y) — line chart
- plt.scatter/figure customization later (Section 08)
- Model functions vis: sigmoid S-shape
- numpy→plot = standard analytic workflow
- Multiple: plt.plot(x, y1); plt.plot(x, y2)
- labels/title optional

In [134]:
# plotting a 2D plot
# x = y
import matplotlib.pyplot as plt

x = np.linspace(-10,10,100)
y = x

plt.plot(x,y)

<Figure size 432x288 with 1 Axes>

In [135]:
# y = x^2
x = np.linspace(-10,10,100)
y = x**2

plt.plot(x,y)

<Figure size 432x288 with 1 Axes>

In [136]:
# y = sin(x)
x = np.linspace(-10,10,100)
y = np.sin(x)

plt.plot(x,y)

<Figure size 432x288 with 1 Axes>

In [137]:
# y = xlog(x)
x = np.linspace(-10,10,100)
y = x * np.log(x)

plt.plot(x,y)

<ipython-input-137-4b3958c08378>:3: RuntimeWarning: invalid value encountered in log
  y = x * np.log(x)


<Figure size 432x288 with 1 Axes>

In [138]:
# sigmoid
x = np.linspace(-10,10,100)
y = 1/(1+np.exp(-x))

plt.plot(x,y)

<Figure size 432x288 with 1 Axes>

### Meshgrids

## 📖 Meshgrid — 2D grid banane ka formula

```python
X, Y = np.meshgrid(x, y)
```

- `meshgrid` — 2 vectors se coordinate grid banata hai
- X me repeating rows, Y me repeating columns
- 3D surface plots / contour plots ke liye base
- `plt.contour`, `ax.plot_surface` isi grid pe kaam karte hain

## 🔬 Deep Dive : Meshgrid — 2D grid

```python
X, Y = np.meshgrid(np.arange(0,10), np.arange(0,10))
# X: x repeated rows | Y: y repeated cols
# grid coordinates for 2D surface plots
```

- meshgrid — 1D x, y → 2D coordinate grids
- X[i,j] = x values repeated rows
- Y[i,j] = y values repeated columns
- Mesh = every (xi, yi) pair — grid of coordinates
- Use: 2D functions `Z = f(X,Y)` surfaces, contour, contourf
- Also: vectorized eval on all grid points
- Interpolation, heatmap plotting
- plt.contourf/plot_surface me jodi
- Evaluate formula at every grid cell without X-loop
- Numpy grid = scientific computation essential

In [57]:
# Meshgrids